In [ ]:
# Needed to import local modules since pwd is notebooks folder instead of project folder
import os
os.chdir('..')

# Candle Outcome Visualisations

In [ ]:
import pandas as pd
from sqlalchemy import engine, text
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.database import get_engine
from src.visualisation.db_visualisation import view_outcomes_wide

In [ ]:
conn = get_engine()

In [ ]:
holding_period = 1000

query = """
    WITH candle_count AS (
        SELECT
            COUNT (DISTINCT candle_id) AS total_candles
        FROM OUTCOMES
    )
    SELECT
        a.return_threshold AS threshold_x,
        b.return_threshold AS threshold_y,
        COUNT(*) AS count_x_before_y,
        total_candles
    FROM outcomes a
    JOIN outcomes b ON a.candle_id = b.candle_id AND (a.return_threshold * b.return_threshold < 0)
    CROSS JOIN candle_count
    WHERE (a.candles_to_hit < b.candles_to_hit
        OR (a.candles_to_hit IS NOT NULL AND b.candles_to_hit IS NULL)) 
        AND a.candles_to_hit <= :holding_period
    GROUP BY threshold_x, threshold_y, total_candles
    ORDER BY threshold_x;
"""

In [ ]:
df = pd.read_sql(text(query), conn, params={"holding_period": holding_period})

In [ ]:
contingency_table = df.pivot_table(index='threshold_x', columns='threshold_y', values='count_x_before_y')

In [ ]:
plt.figure(figsize=(12, 8))
ax = sns.heatmap(contingency_table, annot=True, cmap='viridis')
ax.invert_yaxis()
plt.show()

## Conditional Outcomes 

In [ ]:
number_of_bins = 10
holding_period = 1000

conditional_query = """
    WITH indicator_binned AS (
        SELECT                              -- Discretises the indicators values into bins of specified number
            candle_id,
            indicator_id,
            NTILE(:number_of_bins) OVER (PARTITION BY indicator_id ORDER BY indicator_value) AS percentile_bin
        FROM indicator_values               -- This CTE actually has lookahead bias
        WHERE candle_id IN (SELECT candle_id FROM outcomes)
    ),
    indicator_binned_counts AS (            -- CTE for count of candles in each bin, only for pandas to pull through in one database request
        SELECT                              -- (ie it's not used in the core logic)
            indicator_id,
            percentile_bin,
            COUNT(*) AS bin_candle_count
        FROM indicator_binned
        GROUP BY indicator_id, percentile_bin
    )
    SELECT
        a.return_threshold AS threshold_x,
        b.return_threshold AS threshold_y,
        ib.indicator_id,
        ib.percentile_bin,
        COUNT(*) AS x_hit_before_y,
        ibc.bin_candle_count
    FROM outcomes a
    JOIN indicator_binned ib 
        ON ib.candle_id = a.candle_id
    JOIN outcomes b 
        ON b.candle_id = a.candle_id 
        AND (a.return_threshold * b.return_threshold < 0)
    JOIN indicator_binned_counts ibc 
        ON ibc.percentile_bin = ib.percentile_bin 
        AND ibc.indicator_id = ib.indicator_id
    WHERE 
        (a.candles_to_hit < b.candles_to_hit 
        OR (a.candles_to_hit IS NOT NULL AND b.candles_to_hit IS NULL))
        AND a.candles_to_hit <= :holding_period
    GROUP BY 
        threshold_x, 
        threshold_y, 
        ib.indicator_id, 
        ib.percentile_bin, 
        ibc.bin_candle_count
    ORDER BY threshold_x;
"""

In [ ]:
raw = pd.read_sql(text(conditional_query), conn, params={"number_of_bins": number_of_bins, "holding_period": holding_period})
raw

In [ ]:
# Converting from raw values to probabilities
x_before_y_bin_1 = raw.loc[(raw['indicator_id'] == 1) & (raw['percentile_bin'] == 1)].copy()

In [ ]:
x_before_y_bin_1['probabilities'] = x_before_y_bin_1['x_hit_before_y'] / x_before_y_bin_1['bin_candle_count']

In [ ]:
contingency_table = x_before_y_bin_1.pivot_table(index='threshold_x', columns='threshold_y', values='x_hit_before_y')
contingency_table

# Sense Check

Element-wise x_before_y + y_before_x + unresolved + same_candle_hit should be equal to total candles / number of bins.

In [ ]:
# Constants
number_of_bins = 10
holding_period = 1000

In [ ]:
# Unresolved candles
unresolved_query = """
    WITH indicator_binned AS (
        SELECT                              -- Discretises the indicators values into bins of specified number
            candle_id,
            indicator_id,
            NTILE(:number_of_bins) OVER (PARTITION BY indicator_id ORDER BY indicator_value) AS percentile_bin
        FROM indicator_values               -- This CTE actually has lookahead bias
        WHERE candle_id IN (SELECT candle_id FROM outcomes)
    ),
    indicator_binned_counts AS (            -- CTE for count of candles in each bin, only for pandas to pull through in one database request
        SELECT                              -- (ie it's not used in the core logic)
            indicator_id,
            percentile_bin,
            COUNT(*) AS bin_candle_count
        FROM indicator_binned
        GROUP BY indicator_id, percentile_bin
    )
    SELECT
        a.return_threshold AS threshold_x,
        b.return_threshold AS threshold_y,
        ib.indicator_id,
        ib.percentile_bin,
        COUNT(*) AS x_hit_before_y,
        ibc.bin_candle_count
    FROM outcomes a
    JOIN indicator_binned ib 
        ON ib.candle_id = a.candle_id
    JOIN outcomes b 
        ON b.candle_id = a.candle_id 
        AND (a.return_threshold * b.return_threshold < 0)
    JOIN indicator_binned_counts ibc 
        ON ibc.percentile_bin = ib.percentile_bin 
        AND ibc.indicator_id = ib.indicator_id
    WHERE 
        (a.candles_to_hit IS NULL OR a.candles_to_hit > :holding_period)
        AND (b.candles_to_hit IS NULL OR b.candles_to_hit > :holding_period)
    GROUP BY 
        threshold_x, 
        threshold_y, 
        ib.indicator_id, 
        ib.percentile_bin, 
        ibc.bin_candle_count
    ORDER BY threshold_x;
"""

# Requesting and filtering
unresolved = pd.read_sql(text(unresolved_query), conn, params={"number_of_bins": number_of_bins, "holding_period": holding_period})
unresolved = unresolved.loc[(unresolved['indicator_id'] == 1) & (unresolved['percentile_bin'] == 1)]

In [ ]:
# Same candles hit
same_candles_query = """
    WITH indicator_binned AS (
        SELECT                              -- Discretises the indicators values into bins of specified number
            candle_id,
            indicator_id,
            NTILE(:number_of_bins) OVER (PARTITION BY indicator_id ORDER BY indicator_value) AS percentile_bin
        FROM indicator_values               -- This CTE actually has lookahead bias
        WHERE candle_id IN (SELECT candle_id FROM outcomes)
    ),
    indicator_binned_counts AS (            -- CTE for count of candles in each bin, only for pandas to pull through in one database request
        SELECT                              -- (ie it's not used in the core logic)
            indicator_id,
            percentile_bin,
            COUNT(*) AS bin_candle_count
        FROM indicator_binned
        GROUP BY indicator_id, percentile_bin
    )
    SELECT
        a.return_threshold AS threshold_x,
        b.return_threshold AS threshold_y,
        ib.indicator_id,
        ib.percentile_bin,
        COUNT(*) AS x_hit_before_y,
        ibc.bin_candle_count
    FROM outcomes a
    JOIN indicator_binned ib 
        ON ib.candle_id = a.candle_id
    JOIN outcomes b 
        ON b.candle_id = a.candle_id 
        AND (a.return_threshold * b.return_threshold < 0)
    JOIN indicator_binned_counts ibc 
        ON ibc.percentile_bin = ib.percentile_bin 
        AND ibc.indicator_id = ib.indicator_id
    WHERE 
        a.candles_to_hit = b.candles_to_hit
    GROUP BY 
        threshold_x, 
        threshold_y, 
        ib.indicator_id, 
        ib.percentile_bin, 
        ibc.bin_candle_count
    ORDER BY threshold_x;
"""

In [ ]:
# x_before_y and y_before_x contingencies
x_before_y = raw.loc[(raw['indicator_id'] == 1) & (raw['percentile_bin'] == 1)]

In [ ]:
# Pulling dfs from sql
unresolved = pd.read_sql(text(unresolved_query), conn, params={"number_of_bins": number_of_bins, "holding_period": holding_period})
same_candle = pd.read_sql(text(same_candles_query), conn, params={"number_of_bins": number_of_bins, "holding_period": holding_period})

In [ ]:
# Filter dfs for relevant indicator and bin
x_before_y = raw.loc[(raw['indicator_id'] == 1) & (raw['percentile_bin'] == 1)]
unresolved = unresolved.loc[(unresolved['indicator_id'] == 1) & (unresolved['percentile_bin'] == 1)]
same_candle_hit = same_candle.loc[(same_candle['indicator_id'] == 1) & (same_candle['percentile_bin'] == 1)]

In [ ]:
# Making all tables the same shape for element-wise addition
x_before_y_contingency = x_before_y.pivot_table(index='threshold_x', columns='threshold_y', values='x_hit_before_y')
same_candle_contingency = same_candle_hit.pivot_table(index='threshold_x', columns='threshold_y', values='x_hit_before_y').reindex_like(x_before_y_contingency).fillna(0)
unresolved_contingency = unresolved.pivot_table(index='threshold_x', columns='threshold_y', values='x_hit_before_y').reindex_like(x_before_y_contingency).fillna(0)

In [ ]:
y_before_x_contingency = x_before_y_contingency.T

In [ ]:
element_wise_addition = x_before_y_contingency + y_before_x_contingency + unresolved_contingency + same_candle_contingency
element_wise_addition

# All values shown as 

## Threshold pair visualisation - Fix bin, fix indicator

First, fixing the indicator ID and bin to visualise the threshold pair 'x before y' heatmap

In [ ]:
# Filter df to fix ID and bin
df = raw.copy()
df['probability'] = df['x_hit_before_y'] / df['bin_candle_count']

# Filter to fix indicator ID and percentile bin
df_bin = df.loc[(df['indicator_id'] == 1) & (df['percentile_bin'] == 1)]

In [ ]:
df_contingency = df_bin.pivot_table(values='probability', index='threshold_x', columns='threshold_y')

In [ ]:
fig, axs = plt.subplots(figsize=(12,8))
axs = sns.heatmap(df_contingency)

## Pair Threshold Visualisation 2 - Fix indicator, iterate bin
Visualises X before Y for each bin

In [ ]:
# Filter df to fix ID and bin
df = raw.copy()
df['probability'] = df['x_hit_before_y'] / df['bin_candle_count']


In [ ]:
# Iterate and subplot each bin
total_bins = df['percentile_bin'].unique()

# Get it to scale with the number of bins
fig, axs = plt.subplots(3,4,figsize=(12,8))

axs = axs.flatten()
print(len(axs))

for bin in total_bins:
    df_bin = df.loc[(df['indicator_id'] == 1) & (df['percentile_bin'] == bin)]   # Filter for bin
    df_contingency = df_bin.pivot_table(values='probability', index='threshold_x', columns='threshold_y')
    sns.heatmap(df_contingency, ax=axs[bin-1], cmap='viridis')


In [ ]:
df_bin = df.loc[(df['indicator_id'] == 1) & (df['percentile_bin'] == 1)]
df_contingency = df.pivot_table(values='probability', index='threshold_x', columns=['percentile_bin', 'threshold_y'])
df_contingency


In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df_contingency)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))  # adjust size for your table
sns.heatmap(
    df_contingency,
    annot=True,      # show the actual values in each cell
    fmt='.2f',       # 2 decimal places
    cmap='RdYlGn',   # red = low probability, green = high
    linewidths=0.5
)
plt.tight_layout()
plt.savefig('contingency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Maybe redyellowgren is more intuitive - actualy no, sequential is better, since there is not 'direction' (one isn't more 'profiatlbe', it just depends on the tradesetup
# This is good justification for colours scheme reasoning in the visualisation section

# I think it makes more sense to view them all stretched into file like this, rather than is subplots - this way I really can
# see them on the same references plots and scale.
# Chose this visualisation due to them all having the same scale - maes it very easy to see patterns.

# Colours - the red and green makes it more obvious than the red and black that there's something at percentile bin 5 for RSI. 


## REMEMBER TO INVERT AXES FROM THE START - seems an interesting visualisation for the 4th percentile bin with RSI. I wonder why?? 
# Remember - in Jane Street it's not enough to observe a pattern. You need to answer why.